## Complex model for selecting best candidate

1. on input: data from NER and 10 candidates for each named entity, and ground truth entities with targt DBpedia URI - this comes from trainig dataset
2. Then train the model, and perform NER.
3. on output: metric showing how well the model selects the candidates - on the training dataset, and then on test dataset


In [15]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from fuzzywuzzy import fuzz
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [16]:
json_file_path = "../DjangoApp/NEL_project/NEL_app/Evaluation/candidate_selection_data/candidates_data_collector_20250417_110240.json"

In [17]:

# Load the JSON data
with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Show top-level structure
print("Keys in loaded JSON:")
print(data.keys())

# Access configuration
print("\nConfiguration:")
print(data["configuration"])

# Access one golden annotation entry
print("\nSample Golden Annotation:")
print(json.dumps(data["golden_annotations"][0], indent=2))

# Access one prediction entry and its candidates
print("\nSample Prediction with Candidates:")
print(json.dumps(data["predictions"][0], indent=2))

golden_annotations = data["golden_annotations"]
predictions = data["predictions"]

Keys in loaded JSON:
dict_keys(['name', 'configuration', 'golden_annotations', 'predictions'])

Configuration:
{'dataset_path': './EvaluationDatasets/ace2004_full.json', 'dataset_total_texts': 119, 'dataset_total_mentions': 257, 'ned_knowledge_graph': 'dbpedia'}

Sample Golden Annotation:
{
  "content": "Bandar Seri Begawan 11 15 AFP The United States today Wednesday deemed the order issued by Palestinian President Yasser Arafat for a ceasefire in territories under Palestinian Authority control as a positive gesture but considered that it does not release constitute a release form the terms of the Sharm el Sheikh agreement James Stewart the White House s spokesman in Bandar Seri Begawan the capital of the Sultanate of Brunei which American President Bill Clinton is visiting said of course we positively welcome the announcement aimed at stopping the violence But the important point is that Palestinian and Israeli officials take the right",
  "entities": [
    {
      "entity_label": "Ba

## Candidates Scorers

In [18]:
# Define scorers:
class ContextScorer:
    def __init__(self, model_name="all-MiniLM-L6-v2", round_to=3):
        self.model = SentenceTransformer(model_name)
        self.round_to = round_to

    def score(self, context_text, candidate_comment):
        context_emb = self.model.encode(context_text, convert_to_tensor=True)
        candidate_emb = self.model.encode(candidate_comment, convert_to_tensor=True)
        score = util.pytorch_cos_sim(context_emb, candidate_emb).item()
        return round(score, self.round_to)

class LevenshteinDistanceScorer:
    def __init__(self, round_to=3):
        self.round_to = round_to

    def score(self, label1, label2):
        return round(fuzz.ratio(label1, label2) / 100.0, self.round_to)

class PopularityScorer:
    def __init__(self, round_to=3):
        self.round_to = round_to

    def score_all(self, candidates):
        ref_counts = [c["ref_count"] for c in candidates]
        log_counts = np.log1p(ref_counts)
        min_log, max_log = np.min(log_counts), np.max(log_counts)

        scores = []
        for log_val in log_counts:
            if max_log == min_log:
                scores.append(1.0)
            else:
                normalized = (log_val - min_log) / (max_log - min_log)
                scores.append(round(normalized, self.round_to))
        return scores

# ========== Initialize Scorers ==========
context_scorer = ContextScorer()
levenshtein_scorer = LevenshteinDistanceScorer()
popularity_scorer = PopularityScorer()


In [19]:
# ========== Calculate Scores ==========
scored_data = []

for entry in tqdm(predictions, desc="Scoring predictions"):
    context_text = entry["content"]
    for entity in entry["entities"]:
        entity_label = entity["entity_label"]
        start_position = entity["start_position"]
        end_position = entity["end_position"]
        candidates = entity["candidates"]

        # Popularity scores (batch)
        pop_scores = popularity_scorer.score_all(candidates)

        for i, candidate in enumerate(candidates):
            label = candidate["label"]
            comment = candidate.get("comment", "")

            score_lev = levenshtein_scorer.score(entity_label, label)
            score_ctx = context_scorer.score(context_text, comment)
            score_pop = pop_scores[i]

            scored_data.append({
                "context_text": context_text,
                "entity_label": entity_label,
                "start_position": start_position,
                "end_position": end_position,
                "candidate_label": label,
                "candidate_uri": candidate["uri"],
                "score_levenshtein": score_lev,
                "score_context": score_ctx,
                "score_popularity": score_pop
            })

# ========== Convert to DataFrame for Analysis or Model Input ==========
df = pd.DataFrame(scored_data)

Scoring predictions: 100%|████████████████████████████████████████████████████████████| 119/119 [00:10<00:00, 11.81it/s]


In [21]:
df.head()

,context_text,entity_label,start_position,end_position,candidate_label,candidate_uri,score_levenshtein,score_context,score_popularity
0,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,Bandar Seri Begawan,http://dbpedia.org/resource/Bandar_Seri_Begawan,1.00,0.354,0.839
1,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,"Pusat Bandar, Brunei","http://dbpedia.org/resource/Pusat_Bandar,_Brunei",0.46,0.348,0.289
2,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,"Embassy of the Philippines, Bandar Seri Begawan",http://dbpedia.org/resource/Embassy_of_the_Phi...,0.58,0.450,0.124
3,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,1999 Southeast Asian Games,http://dbpedia.org/resource/1999_Southeast_Asi...,0.27,0.189,0.349
4,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,List of diplomatic missions of Russia,http://dbpedia.org/resource/List_of_diplomatic...,0.21,0.087,1.000


## Neural network model - 3 features

In [24]:

# ========== Prepare Data for the Neural Network with Grouping ==========

def create_best_candidate_flag_grouped(predictions_df, golden_annotations):
    best_candidate_uris = {}
    for gold_entry in golden_annotations:
        for entity in gold_entry['entities']:
            key = (gold_entry['content'], entity['entity_label'], entity['start_position'], entity['end_position'])
            best_candidate_uris[key] = entity['best_candidate_uri']

    predictions_df['is_best_candidate'] = False
    for index, row in predictions_df.iterrows():
        key = (row['context_text'], row['entity_label'], row['start_position'], row['end_position'])
        if key in best_candidate_uris and row['candidate_uri'] == best_candidate_uris[key]:
            predictions_df.loc[index, 'is_best_candidate'] = True
    return predictions_df

df_with_best_flag = create_best_candidate_flag_grouped(df.copy(), golden_annotations)


In [25]:
df_with_best_flag.head()

,context_text,entity_label,start_position,end_position,candidate_label,candidate_uri,score_levenshtein,score_context,score_popularity,is_best_candidate
0,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,Bandar Seri Begawan,http://dbpedia.org/resource/Bandar_Seri_Begawan,1.00,0.354,0.839,True
1,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,"Pusat Bandar, Brunei","http://dbpedia.org/resource/Pusat_Bandar,_Brunei",0.46,0.348,0.289,False
2,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,"Embassy of the Philippines, Bandar Seri Begawan",http://dbpedia.org/resource/Embassy_of_the_Phi...,0.58,0.450,0.124,False
3,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,1999 Southeast Asian Games,http://dbpedia.org/resource/1999_Southeast_Asi...,0.27,0.189,0.349,False
4,Bandar Seri Begawan 11 15 AFP The United State...,Bandar Seri Begawan,0,19,List of diplomatic missions of Russia,http://dbpedia.org/resource/List_of_diplomatic...,0.21,0.087,1.000,False


In [70]:

class GroupedCandidateSelectionDataset(Dataset):
    def __init__(self, dataframe, max_candidates=10):
        self.grouped_data = dataframe.groupby(['context_text', 'entity_label', 'start_position', 'end_position'])
        self.keys = list(self.grouped_data.groups.keys())
        self.max_candidates = max_candidates

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        group = self.grouped_data.get_group(key)
        num_candidates = len(group)

        # Pad or truncate to max_candidates
        features = np.zeros((self.max_candidates, 3), dtype=np.float32)
        target = np.zeros(self.max_candidates, dtype=np.float32)

        scores = group[['score_levenshtein', 'score_context', 'score_popularity']].values
        is_best = group['is_best_candidate'].values.astype(np.float32)

        features[:num_candidates, :] = scores
        target[:num_candidates] = is_best

        # Determine the index of the best candidate (if any) for the target
        best_candidate_index = -1
        if np.any(is_best):
            best_candidate_index = np.where(is_best == 1)[0][0]

        return torch.tensor(features, dtype=torch.float32), torch.tensor(target, dtype=torch.float32), torch.tensor(best_candidate_index, dtype=torch.long)

class GroupedCandidateSelectionNN(nn.Module):
    def __init__(self, num_candidates, input_size):
        super(GroupedCandidateSelectionNN, self).__init__()
        self.fc1 = nn.Linear(num_candidates * input_size, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc3 = nn.Linear(32, num_candidates)
        self.softmax = nn.Softmax(dim=1) # Output probabilities for each candidate

    def forward(self, x):
        # Flatten the input (batch_size, num_candidates, input_size) -> (batch_size, num_candidates * input_size)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = self.softmax(x)
        return x

# ========== Prepare DataLoaders for Grouped Data ==========
MAX_CANDIDATES = 10 # Define the maximum number of candidates to consider

# Create the grouped dataset
grouped_dataset = GroupedCandidateSelectionDataset(df_with_best_flag, max_candidates=MAX_CANDIDATES)


In [71]:

# Split keys for training and testing
train_keys, test_keys = train_test_split(grouped_dataset.keys, test_size=0.2)

def create_dataloader_from_keys(dataset, keys, batch_size):
    sampler = torch.utils.data.SubsetRandomSampler([dataset.keys.index(k) for k in keys])
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler)

train_loader_grouped = create_dataloader_from_keys(grouped_dataset, train_keys, batch_size=32)
test_loader_grouped = create_dataloader_from_keys(grouped_dataset, test_keys, batch_size=32)

# ========== Initialize the Grouped Neural Network and Optimizer ==========
input_size = 3  # Levenshtein, Context, Popularity scores
model_grouped = GroupedCandidateSelectionNN(MAX_CANDIDATES, input_size)
criterion_grouped = nn.CrossEntropyLoss(ignore_index=-1) # Ignore the padding for target
optimizer_grouped = optim.Adam(model_grouped.parameters(), lr=0.001)


In [72]:
# ========== Training the Grouped Neural Network with Accuracy ==========
epochs = 20
for epoch in range(epochs):
    model_grouped.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target, best_index) in enumerate(train_loader_grouped):
        optimizer_grouped.zero_grad()
        output = model_grouped(data)  # Output shape: (batch_size, num_candidates)

        loss = criterion_grouped(output, best_index)
        loss.backward()
        optimizer_grouped.step()

        total_loss += loss.item()

        # Compute accuracy
        predicted = output.argmax(dim=1)         # Predicted candidate index
        correct += (predicted == best_index).sum().item()
        total += best_index.size(0)              # Number of samples in batch

    accuracy = correct / total * 100
    avg_loss = total_loss / len(train_loader_grouped)
    print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')


Epoch 1, Loss: 2.2818, Accuracy: 32.20%
Epoch 2, Loss: 2.2624, Accuracy: 67.32%
Epoch 3, Loss: 2.2257, Accuracy: 71.71%
Epoch 4, Loss: 2.1530, Accuracy: 71.71%
Epoch 5, Loss: 1.9831, Accuracy: 71.71%
Epoch 6, Loss: 1.7924, Accuracy: 71.71%
Epoch 7, Loss: 1.7142, Accuracy: 71.71%
Epoch 8, Loss: 1.6817, Accuracy: 71.71%
Epoch 9, Loss: 1.6843, Accuracy: 71.71%
Epoch 10, Loss: 1.6637, Accuracy: 71.71%
Epoch 11, Loss: 1.6492, Accuracy: 71.71%
Epoch 12, Loss: 1.6690, Accuracy: 71.71%
Epoch 13, Loss: 1.6749, Accuracy: 71.71%
Epoch 14, Loss: 1.6792, Accuracy: 71.71%
Epoch 15, Loss: 1.6675, Accuracy: 71.71%
Epoch 16, Loss: 1.6757, Accuracy: 71.71%
Epoch 17, Loss: 1.6670, Accuracy: 71.71%
Epoch 18, Loss: 1.6799, Accuracy: 71.71%
Epoch 19, Loss: 1.6741, Accuracy: 71.71%
Epoch 20, Loss: 1.6720, Accuracy: 71.71%


In [73]:

# ========== Short Evaluation Process for Grouped Model ==========
model_grouped.eval()
correct_predictions = 0
total_entities = 0

with torch.no_grad():
    for data, target, best_index in test_loader_grouped:
        output = model_grouped(data)
        predicted_index = torch.argmax(output, dim=1)
        # Only consider cases where a best candidate exists in the golden annotations
        mask = (best_index != -1)
        correct_predictions += (predicted_index[mask] == best_index[mask]).sum().item()
        total_entities += mask.sum().item()

accuracy_grouped = correct_predictions / total_entities if total_entities > 0 else 0.0
print(f'Grouped Test Accuracy: {accuracy_grouped:.4f}')


Grouped Test Accuracy: 0.8085


In [74]:
# save the model:
torch.save(model_grouped.state_dict(), "model_grouped.pth")

In [83]:
# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate your model
model_grouped = GroupedCandidateSelectionNN(MAX_CANDIDATES, input_size)  # Replace with your actual model and arguments
model_grouped.load_state_dict(torch.load("model_grouped.pth"))
model_grouped.eval()


GroupedCandidateSelectionNN(
  (fc1): Linear(in_features=30, out_features=64, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.2, inplace=False)
  (fc3): Linear(in_features=32, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)